In [ ]:
# 셀 1: 패키지 설치
!pip install -q rouge-score openai transformers peft bitsandbytes accelerate bert-score
print('패키지 설치 완료')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.3 MB/s eta 0:00:00
패키지 설치 완료


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print('로그인 완료!')

로그인 완료!


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from huggingface_hub import HfApi
import os

api = HfApi()
REPO_ID = "subaksu/re-mind"

CHECKPOINT = '/content/drive/MyDrive/remind_finetune/qwen_finetuned'
print(os.path.exists(CHECKPOINT))  # True 나와야 함

api.upload_folder(
    folder_path=CHECKPOINT,
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["checkpoint-*"],
)
print("업로드 완료!")

Mounted at /content/drive
True


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.4kB /  162MB            

No files have been modified since last commit. Skipping to prevent empty commit.


업로드 완료!


In [ ]:
!pip install -q pyngrok bitsandbytes>=0.46.1

from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
print(f"외부 URL: {public_url}")

외부 URL: NgrokTunnel: "https://lankiness-revered-skilled.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from pyngrok import ngrok

# 기존 터널 모두 종료
ngrok.kill()

# 다시 연결
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
print(f"외부 URL: {public_url}")

외부 URL: NgrokTunnel: "https://lankiness-revered-skilled.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
import json
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import uvicorn
import threading
from google.colab import drive

drive.mount('/content/drive')

# 모델 로드
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
CHECKPOINT = '/content/drive/MyDrive/remind_finetune/qwen_finetuned/checkpoint-8442'
SYSTEM_PROMPT = "당신은 앱 사용자의 생활 로그와 PHQ-9 분석 데이터를 바탕으로 의사용 임상 요약 리포트를 생성하는 AI입니다. 반드시 주요 증상, 위험요인, 개선요인 3가지 섹션으로만 작성하고 진료 권고나 결론은 포함하지 마세요."

print("모델 로드 중...")

# torch.load 패치
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load
def patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return torch._original_load(*args, **kwargs)
torch.load = patched_torch_load

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='cuda:0',
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(model, CHECKPOINT)
model.eval()
print("모델 로드 완료!")

# FastAPI 앱
app = FastAPI()

class ReportRequest(BaseModel):
    input_data: dict

@app.post("/generate")
async def generate(req: ReportRequest):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(req.input_data, ensure_ascii=False)}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return {"output": result}

# 백그라운드에서 서버 실행
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print(f"Qwen 서버 실행 중!")
print(f"외부 URL: {public_url}/generate")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
모델 로드 중...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

모델 로드 완료!
Qwen 서버 실행 중!
외부 URL: NgrokTunnel: "https://lankiness-revered-skilled.ngrok-free.dev" -> "http://localhost:8000"/generate


In [ ]:
import requests
resp = requests.post(
    "https://lankiness-revered-skilled.ngrok-free.dev/generate",
    json={"input_data": {"diary_logs": [], "phq9_analysis": {"total_score": 5, "risk_level": "경미한위험", "phq9_slots": []}}},
    timeout=60
)
print(resp.status_code)
print(resp.json())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


INFO:     34.124.202.96:0 - "POST /generate HTTP/1.1" 200 OK
200
{'output': '**주요 증상:** 내담자는 경미한 우울 증상을 보이며, 일상적인 활동에서 간헐적으로 기분 저하를 경험하고 있다. 최근의 감정적 불안정성과 함께, 간헐적인 자해 사고가 관찰되고 있으며, 이는 내담자의 정서적 안정에 부정적인 영향을 미치고 있다.\n\n**위험요인:** 내담자는 사회적 격리감과 외로움을 느끼며, 이는 우울증의 심화 요인으로 작용할 수 있다. 또한, 자해 사고와 관련된 위험 요소가 존재하며, 이러한 행동은 내담자의 정신 건강에 심각한 위협으로 작용할 가능성이 있다.\n\n**개선요인:** 내담자는 최근 긍정적인 변화를 경험하고 있으며, 친구와의 교류가 증진되어 정서적 지지를 받고 있다. 또한, 상담을 통해 자신의 감정을 인식하고 관리하는 능력을 향상시키려는 노력을 보이고 있어, 이는 내담자가 자신의 상태를 개선하는 데 도움이 될 수 있는 요소로 작용하고 있다.'}


In [ ]:
# 셀 2: 설정
import os, json, random, gc
import torch
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive

drive.mount('/content/drive')

BASE_MODEL_NAME      = 'Qwen/Qwen2.5-7B-Instruct'
FINETUNED_CHECKPOINT = '/content/drive/MyDrive/remind_finetune/qwen_finetuned/checkpoint-8442'
DATA_PATH            = '/content/drive/MyDrive/remind_finetune/final_train.jsonl'
OUTPUT_DIR           = '/content/drive/MyDrive/remind_finetune'
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = '당신은 앱 사용자의 생활 로그와 PHQ-9 분석 데이터를 바탕으로 의사용 임상 요약 리포트를 생성하는 AI입니다. 반드시 주요 증상, 위험요인, 개선요인 3가지 섹션으로만 작성하고 진료 권고나 결론은 포함하지 마세요.'

print('설정 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
설정 완료


In [ ]:
# 셀 3: 테스트셋 추출
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    all_samples = [json.loads(line) for line in f if line.strip()]

print(f'전체 샘플 수: {len(all_samples)}개')

random.seed(42)
test_candidates = all_samples[-5000:]
test_raw = random.sample(test_candidates, 30)

TEST_SAMPLES = []
for raw in test_raw:
    msgs = raw['messages']
    user_content = json.loads(msgs[1]['content'])
    asst_content = json.loads(msgs[2]['content'])
    TEST_SAMPLES.append({
        'input': user_content,
        'ground_truth': asst_content['output']
    })

print(f'테스트셋 추출 완료: {len(TEST_SAMPLES)}개')
print(f'샘플 1 ground_truth: {TEST_SAMPLES[0]["ground_truth"][:80]}...')

전체 샘플 수: 71102개
테스트셋 추출 완료: 30개
샘플 1 ground_truth: 주요 증상: 환자는 주된 감정으로 '행복'을(를) 보고하였으며, PHQ-9 총점 8점으로 '경미한위험 (가벼운 우울)' 수준에 해당함. 주요 호...


In [ ]:
# 셀 4: 함수 정의
import re

def load_base_model():
    print('베이스 모델 로드 중...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_enable_fp32_cpu_offload=True
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        max_memory={0: '38GiB', 'cpu': '16GiB'},
    )
    print('베이스 모델 로드 완료!')
    return model, tokenizer

def load_finetuned_model():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'사용 가능 메모리: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB')
    print('파인튜닝 모델 로드 중...')
    original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return original_load(*args, **kwargs)
    torch.load = patched_load
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map='cuda:0',
        trust_remote_code=True,
    )
    model.load_adapter(FINETUNED_CHECKPOINT)
    for name, param in model.named_parameters():
        if param.device.type == 'cpu':
            param.data = param.data.to('cuda:0')
    print('파인튜닝 모델 로드 완료!')
    return model, tokenizer

def generate_report(model, tokenizer, sample_input):
    model.eval()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': json.dumps(sample_input, ensure_ascii=False)}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def normalize(text):
    text = re.sub(r'\*+', '', text)
    text = re.sub(r'^-\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ROUGE
def calc_rouge(prediction, reference):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = scorer.score(normalize(reference), normalize(prediction))
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure,
    }

# BERTScore (배치 계산용 - 모든 샘플 모은 후 한번에 계산)
def calc_bertscore_batch(predictions, references):
    P, R, F1 = bert_score(
        predictions, references,
        model_type='klue/roberta-base',
        lang='ko',
        num_layers=12,
        verbose=False
    )
    return [{'precision': p.item(), 'recall': r.item(), 'f1': f.item()}
            for p, r, f in zip(P, R, F1)]

# 형식 준수

# LLM-as-a-Judge (Clinical_Accuracy, Conciseness, Formatting + Hallucination)
def llm_judge(report, sample_input):
    diary_text = '\n'.join([
        f"{log['date']}: {log.get('diary', '')}"
        for log in sample_input['diary_logs']
    ])
    phq9_text = '\n'.join([
        f"  PHQ-9 {slot['item_no']}번 ({slot['symptom']}): {slot['score']}점 - {slot['evidence']}"
        for slot in sample_input['phq9_analysis']['phq9_slots']
    ])
    prompt = f"""당신은 정신건강의학과 전문의입니다.
아래 환자 데이터를 바탕으로 생성된 임상 리포트를 평가해주세요.

[환자 원본 데이터]
PHQ-9 총점: {sample_input['phq9_analysis']['total_score']}점
위험도: {sample_input['phq9_analysis']['risk_level']}
PHQ-9 슬롯:
{phq9_text}
일기 요약:
{diary_text[:500]}

[평가 대상 리포트]
{report}

평가 기준 (각 1~5점):
1. Clinical_Accuracy: 환자 데이터에 근거한 사실만 기술했는가?
2. Conciseness: 불필요한 서술 없이 핵심 정보만 담았는가?
3. Formatting: 주요증상/위험요인/개선요인 구조가 명확한가?
4. Hallucination: 환자 데이터에 없는 내용을 지어냈는가? (5=없음/정확, 1=심각한 hallucination)

반드시 JSON으로만 답변:
{{"Clinical_Accuracy": 점수, "Conciseness": 점수, "Formatting": 점수, "Hallucination": 점수, "총점": 평균, "코멘트": "설명"}}"""
    res = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300,
        temperature=0.3,
    )
    raw = res.choices[0].message.content.strip()
    try:
        js = raw[raw.find('{'):raw.rfind('}')+1]
        result = json.loads(js)
        result['총점'] = np.mean([result['Clinical_Accuracy'], result['Conciseness'],
                                   result['Formatting'], result['Hallucination']])
        return result
    except:
        return {'Clinical_Accuracy': 0, 'Conciseness': 0, 'Formatting': 0,
                'Hallucination': 0, '총점': 0, '코멘트': '파싱 실패'}

print('함수 정의 완료')

함수 정의 완료


In [ ]:
# 셀 5-A: 베이스 Qwen 평가
results_base = {'outputs': [], 'rouge': [], 'bertscore': [], 'judge': []}

print('=' * 55)
print('  베이스 Qwen 7B (파인튜닝 전) 평가')
print('=' * 55)

base_model, base_tokenizer = load_base_model()

for i, sample in enumerate(TEST_SAMPLES, 1):
    print(f'[샘플 {i}/{len(TEST_SAMPLES)}]')
    output       = generate_report(base_model, base_tokenizer, sample['input'])
    rouge        = calc_rouge(output, sample['ground_truth'])
    judge        = llm_judge(output, sample['input'])
    results_base['outputs'].append(output)
    results_base['rouge'].append(rouge)
    results_base['judge'].append(judge)
    print(f"  ROUGE-L: {rouge['rougeL']:.4f} | 판사점수: {judge['총점']:.1f}/5")

print('\nBERTScore 계산 중...')
bs = calc_bertscore_batch(results_base['outputs'],
                          [s['ground_truth'] for s in TEST_SAMPLES])
results_base['bertscore'] = bs
print(f"BERTScore F1 평균: {np.mean([b['f1'] for b in bs]):.4f}")

del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

with open(f'{OUTPUT_DIR}/eval_base.json', 'w', encoding='utf-8') as f:
    json.dump(results_base, f, ensure_ascii=False, indent=2)

print(f'\n사용 가능 메모리: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB')
print('베이스 Qwen 평가 완료!')

  베이스 Qwen 7B (파인튜닝 전) 평가
베이스 모델 로드 중...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

베이스 모델 로드 완료!
[샘플 1/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 2/30]
  ROUGE-L: 0.1250 | 판사점수: 4.5/5
[샘플 3/30]
  ROUGE-L: 0.0000 | 판사점수: 3.5/5
[샘플 4/30]
  ROUGE-L: 0.0000 | 판사점수: 4.0/5
[샘플 5/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 6/30]
  ROUGE-L: 0.0000 | 판사점수: 4.0/5
[샘플 7/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 8/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 9/30]
  ROUGE-L: 0.3571 | 판사점수: 4.5/5
[샘플 10/30]
  ROUGE-L: 0.0000 | 판사점수: 3.5/5
[샘플 11/30]
  ROUGE-L: 0.0000 | 판사점수: 4.8/5
[샘플 12/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 13/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 14/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 15/30]
  ROUGE-L: 0.0870 | 판사점수: 4.2/5
[샘플 16/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 17/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 18/30]
  ROUGE-L: 0.0400 | 판사점수: 4.5/5
[샘플 19/30]
  ROUGE-L: 0.2222 | 판사점수: 4.5/5
[샘플 20/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 21/30]
  ROUGE-L: 0.0625 | 판사점수: 4.5/5
[샘플 22/30]
  ROUGE-L: 0.0000 | 판사점수: 4.2/5
[샘플 23/30]
  ROUGE-L: 0.1176 | 판사점수: 4.2/5
[샘플 24

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 평균: 0.6463

사용 가능 메모리: 39.0GB
베이스 Qwen 평가 완료!


In [ ]:
# 셀 5-B: 파인튜닝 Qwen 평가
results_finetuned = {'outputs': [], 'rouge': [], 'bertscore': [], 'judge': []}

print('=' * 55)
print('  파인튜닝 Qwen 7B 평가')
print('=' * 55)

ft_model, ft_tokenizer = load_finetuned_model()

for i, sample in enumerate(TEST_SAMPLES, 1):
    print(f'[샘플 {i}/{len(TEST_SAMPLES)}]')
    output       = generate_report(ft_model, ft_tokenizer, sample['input'])
    rouge        = calc_rouge(output, sample['ground_truth'])
    judge        = llm_judge(output, sample['input'])
    results_finetuned['outputs'].append(output)
    results_finetuned['rouge'].append(rouge)
    results_finetuned['judge'].append(judge)
    print(f"  ROUGE-L: {rouge['rougeL']:.4f} | 판사점수: {judge['총점']:.1f}/5")

print('\nBERTScore 계산 중...')
bs = calc_bertscore_batch(results_finetuned['outputs'],
                          [s['ground_truth'] for s in TEST_SAMPLES])
results_finetuned['bertscore'] = bs
print(f"BERTScore F1 평균: {np.mean([b['f1'] for b in bs]):.4f}")

del ft_model, ft_tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

with open(f'{OUTPUT_DIR}/eval_finetuned.json', 'w', encoding='utf-8') as f:
    json.dump(results_finetuned, f, ensure_ascii=False, indent=2)

print(f'\n사용 가능 메모리: {torch.cuda.mem_get_info()[0]/1024**3:.1f}GB')
print('파인튜닝 Qwen 평가 완료!')

  파인튜닝 Qwen 7B 평가
사용 가능 메모리: 39.0GB
파인튜닝 모델 로드 중...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

파인튜닝 모델 로드 완료!
[샘플 1/30]
  ROUGE-L: 0.1538 | 판사점수: 4.5/5
[샘플 2/30]
  ROUGE-L: 0.3810 | 판사점수: 4.5/5
[샘플 3/30]
  ROUGE-L: 0.2105 | 판사점수: 4.5/5
[샘플 4/30]
  ROUGE-L: 0.2667 | 판사점수: 4.5/5
[샘플 5/30]
  ROUGE-L: 0.1538 | 판사점수: 4.5/5
[샘플 6/30]
  ROUGE-L: 0.2000 | 판사점수: 4.5/5
[샘플 7/30]
  ROUGE-L: 0.3000 | 판사점수: 4.5/5
[샘플 8/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 9/30]
  ROUGE-L: 0.3750 | 판사점수: 3.5/5
[샘플 10/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 11/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 12/30]
  ROUGE-L: 0.3750 | 판사점수: 4.5/5
[샘플 13/30]
  ROUGE-L: 0.2105 | 판사점수: 4.5/5
[샘플 14/30]
  ROUGE-L: 0.3158 | 판사점수: 4.5/5
[샘플 15/30]
  ROUGE-L: 0.4000 | 판사점수: 4.5/5
[샘플 16/30]
  ROUGE-L: 0.3000 | 판사점수: 4.5/5
[샘플 17/30]
  ROUGE-L: 0.0000 | 판사점수: 4.5/5
[샘플 18/30]
  ROUGE-L: 0.6000 | 판사점수: 4.5/5
[샘플 19/30]
  ROUGE-L: 0.2105 | 판사점수: 4.5/5
[샘플 20/30]
  ROUGE-L: 0.5000 | 판사점수: 4.5/5
[샘플 21/30]
  ROUGE-L: 0.3750 | 판사점수: 4.2/5
[샘플 22/30]
  ROUGE-L: 0.2500 | 판사점수: 4.5/5
[샘플 23/30]
  ROUGE-L: 0.2857 | 판사점수: 4.5/5
[샘플 2

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1 평균: 0.6880

사용 가능 메모리: 39.0GB
파인튜닝 Qwen 평가 완료!


In [ ]:
# 셀 6: 결과 출력
with open(f'{OUTPUT_DIR}/eval_base.json', 'r', encoding='utf-8') as f:
    results_base = json.load(f)
with open(f'{OUTPUT_DIR}/eval_finetuned.json', 'r', encoding='utf-8') as f:
    results_finetuned = json.load(f)

print('\n' + '='*65)
print('  Re:Mind 파인튜닝 성능 평가 결과 (v3)')
print('='*65)

print('\n[ROUGE Score]')
for metric in ['rouge1', 'rouge2', 'rougeL']:
    base_avg = np.mean([r[metric] for r in results_base['rouge']])
    ft_avg   = np.mean([r[metric] for r in results_finetuned['rouge']])
    diff  = ft_avg - base_avg
    arrow = '▲' if diff > 0 else '▼'
    print(f'  {metric.upper():<10} 베이스: {base_avg:.4f} | 파인튜닝: {ft_avg:.4f} | {arrow}{abs(diff):.4f}')

print('\n[BERTScore]')
for metric in ['precision', 'recall', 'f1']:
    base_avg = np.mean([b[metric] for b in results_base['bertscore']])
    ft_avg   = np.mean([b[metric] for b in results_finetuned['bertscore']])
    diff  = ft_avg - base_avg
    arrow = '▲' if diff > 0 else '▼'
    print(f'  {metric.upper():<10} 베이스: {base_avg:.4f} | 파인튜닝: {ft_avg:.4f} | {arrow}{abs(diff):.4f}')

print('\n[LLM-as-a-Judge]')
for criterion in ['Clinical_Accuracy', 'Conciseness', 'Formatting', 'Hallucination', '총점']:
    base_avg = np.mean([j[criterion] for j in results_base['judge']])
    ft_avg   = np.mean([j[criterion] for j in results_finetuned['judge']])
    diff  = ft_avg - base_avg
    arrow = '▲' if diff > 0 else '▼'
    print(f'  {criterion:<20} 베이스: {base_avg:.2f}/5 | 파인튜닝: {ft_avg:.2f}/5 | {arrow}{abs(diff):.2f}')

print('\n[샘플 1 출력 비교]')
print('\n--- 베이스 Qwen ---')
print(results_base['outputs'][0][:400])
print('\n--- 파인튜닝 Qwen ---')
print(results_finetuned['outputs'][0][:400])

with open(f'{OUTPUT_DIR}/eval_results_final_v3.json', 'w', encoding='utf-8') as f:
    json.dump({'base': results_base, 'finetuned': results_finetuned}, f, ensure_ascii=False, indent=2)
print('\n결과 저장 완료: eval_results_final_v3.json')


  Re:Mind 파인튜닝 성능 평가 결과 (v3)

[ROUGE Score]
  ROUGE1     베이스: 0.0450 | 파인튜닝: 0.2746 | ▲0.2296
  ROUGE2     베이스: 0.0077 | 파인튜닝: 0.1465 | ▲0.1388
  ROUGEL     베이스: 0.0413 | 파인튜닝: 0.2529 | ▲0.2116

[BERTScore]
  PRECISION  베이스: 0.6325 | 파인튜닝: 0.6647 | ▲0.0322
  RECALL     베이스: 0.6610 | 파인튜닝: 0.7136 | ▲0.0526
  F1         베이스: 0.6463 | 파인튜닝: 0.6880 | ▲0.0417

[LLM-as-a-Judge]
  Clinical_Accuracy    베이스: 3.80/5 | 파인튜닝: 3.93/5 | ▲0.13
  Conciseness          베이스: 3.90/5 | 파인튜닝: 4.00/5 | ▲0.10
  Formatting           베이스: 4.93/5 | 파인튜닝: 4.93/5 | ▼0.00
  Hallucination        베이스: 4.67/5 | 파인튜닝: 4.70/5 | ▲0.03
  총점                   베이스: 4.33/5 | 파인튜닝: 4.39/5 | ▲0.07

[샘플 1 출력 비교]

--- 베이스 Qwen ---
주요 증상:
- 스트레스와 틱장애로 인한 불안감과 불행감이 지속되고 있습니다.
- 집중력 저하: 일기를 통해 집중력 문제를 자주 언급하고 있습니다.
- 수면 장애: 간헐적으로 수면 시작 시간이 늦고 수면 시간이 짧은 것으로 나타났습니다.

위험요인:
- 경미한 수면 장애: 수면 시작 시간이 늦게 늦춰지고, 수면 시간이 줄어들어 있습니다.
- 부정적 인식: 자신에 대한 부정적인 생각이 간헐적으로 나타납니다.
- 피로감: 미미한 수준의 피로감을 경험하고 있습니다.

개선요인:
- 약물 복용 시 약물 반응이 긍정적이며, 복약 후 안정감을 느